# Qwen3.5-9B QLoRA: checkpoint comparison across three seeds

- Train seeds 42, 123, and 2026 sequentially on one GPU; every seed starts from a fresh quantized base model.
- Reshuffle unique questions with each run seed into approximately 80/10/10 train/validation/test sets; keep all rows for a question in one set.
- Keep notebook 7's loss-based LR scheduling and early stopping for every selection rule.
- At each quarter-epoch validation, independently preserve adapters selected by lowest loss, pooled exact accuracy, question-averaged exact accuracy, and question-averaged criterion F1.
- Named best-adapter folders are separate from Trainer's rotating checkpoints.
- Evaluate every preserved selection and the disabled-adapter baseline on the held-out test set after each seed finishes training.
- Display all test metrics in a baseline/QLoRA pair for each seed, then average each model across seeds for the lowest-validation-loss QLoRA checkpoint. Preserve differences from baseline and sample standard deviations in CSV exports.
- Inputs, rubric labels, LoRA rank/alpha, and greedy non-thinking decoding are shared by all selection rules.
- Dependency setup runs automatically. Restart the kernel if packages were updated before continuing training. GPU training is gated by `RUN_TRAINING`.


## Dependencies and compiler setup

- This cell always installs the training dependencies and attempts the optional DeltaNet kernels.
- Installation uses the current notebook kernel's Python environment. Restart the kernel if packages were updated before continuing training.
- CUDA compiler/headers can be installed with NVIDIA pip wheels; GCC/G++ must already exist on the host or cluster.
- Optional DeltaNet kernel failures leave the PyTorch fallback.
- References: [Qwen3.5](https://huggingface.co/docs/transformers/model_doc/qwen3_5), [TRL SFT](https://huggingface.co/docs/trl/sft_trainer).


In [1]:
import os
import shutil
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, distribution
from pathlib import Path

subprocess.run([
    sys.executable, "-m", "pip", "install", "-U", "ipywidgets", "torch",
    "transformers", "accelerate", "peft", "datasets", "bitsandbytes", "trl",
    "sentencepiece", "pandas", "scikit-learn", "ninja", "packaging",
    "huggingface_hub", "python-dotenv", "setuptools", "wheel",
    "cuda-toolkit[nvcc,cudart,cccl,crt,nvvm]>=13.0,<13.1",
], check=True)

# Prefer the pip compiler when installed, otherwise use the existing cluster/Conda nvcc.
try:
    nvcc_dist = distribution("nvidia-cuda-nvcc")
    nvcc = next((
        Path(nvcc_dist.locate_file(file)).resolve()
        for file in nvcc_dist.files or [] if str(file).endswith("/bin/nvcc")
    ), None)
except PackageNotFoundError:
    nvcc = None
if nvcc is None or not nvcc.is_file():
    nvcc_on_path = shutil.which("nvcc")
    nvcc = Path(nvcc_on_path).resolve() if nvcc_on_path else None
if nvcc is not None:
    cuda_home = nvcc.parent.parent
    os.environ["CUDA_HOME"] = str(cuda_home)
    os.environ["PATH"] = str(nvcc.parent) + os.pathsep + os.environ.get("PATH", "")
    cuda_include = cuda_home / "include"
    if not cuda_include.is_dir():
        cuda_include = cuda_home / "targets" / "x86_64-linux" / "include"
    os.environ["CPATH"] = str(cuda_include) + os.pathsep + os.environ.get("CPATH", "")
os.environ.setdefault("MAX_JOBS", "1")
cc = os.environ.get("CC") or shutil.which("gcc") or shutil.which("clang")
if cc:
    os.environ["CC"] = cc
if system_cxx := shutil.which("g++"):
    os.environ["CXX"] = system_cxx

for package_args in (["flash-linear-attention[cuda]"], ["causal-conv1d", "--no-build-isolation"]):
    result = subprocess.run([sys.executable, "-m", "pip", "install", "-U", *package_args], check=False)
    if result.returncode:
        print(f"Optional install failed: {package_args[0]}; using the PyTorch fallback.")
print("Dependency installation finished. Restart the kernel if packages were updated before continuing training.")


  Using cached huggingface_hub-2.1.1-py3-none-any.whl.metadata (16 kB)


  Using cached httpx2-2.13.1-py3-none-any.whl.metadata (9.8 kB)
  Using cached httpcore2-2.13.1-py3-none-any.whl.metadata (26 kB)


  Using cached truststore-0.10.4-py3-none-any.whl.metadata (4.4 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/12.6 MB ? eta -:--:--

   ━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━━━━━━━━ 6.3/12.6 MB 32.2 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 31.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 927.2/927.2 kB 8.2 MB/s  0:00:00


...

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0/2 [transformers]

  Attempting uninstall: trl
    Found existing installation: trl 1.14.1
    Uninstalling trl-1.14.1:
      Successfully uninstalled trl-1.14.1
   ━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━ 1/2 [trl]

   ━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━ 1/2 [trl]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [trl]


Dependency installation finished. Restart the kernel if packages were updated before continuing training.


## Imports and hardware

- One CUDA GPU is required for model loading and QLoRA training.
- Dataset and tokenizer checks can run on CPU.

In [2]:
import ast
import csv
import gc
import hashlib
import importlib.util
import json
import math
import random
import re
from collections import defaultdict
from datetime import datetime, timezone
from importlib.metadata import version

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from huggingface_hub import login, upload_folder
from IPython.display import display
from peft import LoraConfig, get_peft_model
from sklearn.metrics import accuracy_score, cohen_kappa_score, precision_recall_fscore_support
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, EarlyStoppingCallback, set_seed
from trl import SFTConfig, SFTTrainer
from tqdm.auto import tqdm

from dotenv import load_dotenv

load_dotenv()
hf_token = os.environ.get("HF_TOKEN")

print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available(), "| Compiler:", cc)
for package in ("causal_conv1d", "fla"):
    print(f"{package} installed:", importlib.util.find_spec(package) is not None)
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f"GPU: {gpu.name} | VRAM: {gpu.total_memory / 1024**3:.1f} GiB")


PyTorch: 2.14.1+cu130 | CUDA: True | Compiler: /opt/apps/software/GCCcore/13.3.0/bin/gcc
causal_conv1d installed: True
fla installed: True
GPU: NVIDIA L40S | VRAM: 44.4 GiB


## Configuration

- `TRAINING_SEEDS` controls both each run's question split and its training randomness.
- Rank 48 / alpha 96 and notebook 7's loss-based training controls are shared by all runs.
- `CHECKPOINT_SELECTIONS` independently tracks all four selection rules; none changes the LR schedule or early stopping.
- `MAX_SEQ_LENGTH=None` sizes sequences from training/validation examples without truncating labels.
- When `RUN_TRAINING=True`, each Run All creates a new timestamped experiment directory under `OUTPUT_DIR`.
- Test evaluation compares every selection against a baseline pass for each seed. No test metric chooses a checkpoint or upload.
- Set `RUN_TEST_EVALUATION=False` to train/save without test passes. Validation generation remains enabled.
- Upload is optional and targets one explicitly configured seed/selection.


In [3]:
MODEL_ID = "Qwen/Qwen3.5-9B"
DATASET_PATH = "./training_data.csv"
OUTPUT_DIR = "./qwen35-9b-grading-checkpoint-comparison"
RUN_TRAINING = True
RUN_TEST_EVALUATION = True
RUN_HF_UPLOAD = False
HF_REPO_ID = "SmuFypTeam5/GradingQlora"
UPLOAD_SEED = 42
UPLOAD_SELECTION = "best_loss"
TRAINING_SEEDS = [42, 123, 2026]
MAX_SEQ_LENGTH = None

LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05
LEARNING_RATE = 5e-5
WEIGHT_DECAY = 0.05
NUM_EPOCHS = 3
TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
VALIDATIONS_PER_EPOCH = 4
LR_REDUCTION_FACTOR = 0.5
LR_PATIENCE = 1
MIN_LEARNING_RATE = 1e-6
EARLY_STOPPING_PATIENCE = 4
EARLY_STOPPING_THRESHOLD = 0.0
GENERATION_MIN_TOKENS = 128
GENERATION_TOKENS_PER_CRITERION = 8

# All rules inspect the same validation checkpoints from one trajectory per seed.
CHECKPOINT_SELECTIONS = {
    "best_loss": ("eval_loss", False),
    "best_exact_accuracy": ("eval_grading_exact_match_accuracy", True),
    "best_question_exact_accuracy": ("eval_grading_question_macro_exact_match_accuracy", True),
    "best_question_f1": ("eval_grading_question_macro_criterion_f1", True),
}
if not TRAINING_SEEDS or any(type(seed) is not int or seed < 0 for seed in TRAINING_SEEDS):
    raise ValueError("TRAINING_SEEDS must contain nonnegative integer seeds.")
if len(set(TRAINING_SEEDS)) != len(TRAINING_SEEDS):
    raise ValueError("Use distinct training seeds.")


## Load the dataset

- Require prompt fields, binary labels, criterion maxima, and final-score metadata.
- Keep blank student answers empty and preserve literal `NA` text.

In [4]:
required_columns = [
    "question", "desired_answer", "student_answer", "criteria[]",
    "criteria_score[]", "criteria_max_score[]", "student_score",
]
df = pd.read_csv(DATASET_PATH, keep_default_na=False)
missing = sorted(set(required_columns) - set(df.columns))
if missing:
    raise ValueError(f"Missing required columns: {missing}")
if df.empty:
    raise ValueError("The dataset is empty.")
print("Rows:", len(df))
display(df.head())

Rows: 3778


,id,question,desired_answer,student_answer,criteria[],criteria_score[],criteria_max_score[],student_score
0,1.1,What is the role of a prototype program in pro...,To simulate the behaviour of portions of the d...,High risk problems are address in the prototyp...,"[""Answer mentions simulating behaviour of port...",[0],[1],0
1,1.1,What is the role of a prototype program in pro...,To simulate the behaviour of portions of the d...,To simulate portions of the desired final prod...,"[""Answer mentions simulating behaviour of port...",[1],[1],1
2,1.1,What is the role of a prototype program in pro...,To simulate the behaviour of portions of the d...,A prototype program simulates the behaviors of...,"[""Answer mentions simulating behaviour of port...",[1],[1],1
3,1.1,What is the role of a prototype program in pro...,To simulate the behaviour of portions of the d...,Defined in the Specification phase a prototype...,"[""Answer mentions simulating behaviour of port...",[1],[1],1
4,1.1,What is the role of a prototype program in pro...,To simulate the behaviour of portions of the d...,It is used to let the users have a first idea ...,"[""Answer mentions simulating behaviour of port...",[0],[1],0


## Parse and validate rows

- Accept JSON/Python arrays and legacy quoted bracket arrays.
- Use label counts and `Answer` boundaries to preserve commas inside legacy criteria.
- Require nonempty criteria, aligned arrays, binary labels, and finite nonnegative score metadata.
- Keep recorded final scores even if they differ from the sum of criterion labels.

In [5]:
def parse_array(value, expected_count=None):
    """Parse structured arrays or legacy bracket text without losing criterion commas."""
    items = value if isinstance(value, list) else None
    if items is None:
        if not isinstance(value, str):
            raise TypeError(f"Expected an array or string, got {type(value).__name__}.")
        text = value.strip()
        # Unwrap quoted JSON/Python strings before handling legacy unquoted entries.
        for _ in range(3):
            try:
                parsed = json.loads(text)
            except json.JSONDecodeError:
                try:
                    parsed = ast.literal_eval(text)
                except (ValueError, SyntaxError):
                    break
            if isinstance(parsed, list):
                items = parsed
                break
            if not isinstance(parsed, str):
                raise ValueError(f"Expected a list, got {value!r}.")
            text = parsed.strip()
        if items is None:
            if not (text.startswith("[") and text.endswith("]")):
                raise ValueError(f"Expected a bracketed list, got {value!r}.")
            inner = text[1:-1].strip()
            if not inner:
                items = []
            elif expected_count == 1:
                items = [inner]
            elif re.search(r",\s*(?=Answer\s)", inner):
                items = re.split(r",\s*(?=Answer\s)", inner)
            else:
                items = next(csv.reader([inner], skipinitialspace=True))
            items = [item.strip() for item in items]
    if expected_count is not None and len(items) != expected_count:
        raise ValueError(f"Expected {expected_count} entries, parsed {len(items)}: {value!r}")
    return items


def parse_binary_scores(value):
    values = parse_array(value)
    if any(type(x) not in (int, str) or str(x) not in ("0", "1") for x in values):
        raise ValueError(f"Expected binary integers, got {value!r}")
    return [int(x) for x in values]


def parse_score_metadata(value):
    values = parse_array(value)
    if any(isinstance(x, bool) for x in values):
        raise ValueError("Criterion maxima cannot be booleans.")
    return [float(x) for x in values]


records = []
errors = []
for index, row in enumerate(df.to_dict("records")):
    try:
        scores = parse_binary_scores(row["criteria_score[]"])
        criteria = parse_array(row["criteria[]"], expected_count=len(scores))
        maxima = parse_score_metadata(row["criteria_max_score[]"])
        final_score = float(row["student_score"])
        if not criteria or any(not isinstance(c, str) or not c.strip() for c in criteria):
            raise ValueError("Criteria must be nonempty strings.")
        if len(criteria) != len(maxima):
            raise ValueError("Criterion, label, and maximum-score counts must match.")
        if any(not math.isfinite(x) or x < 0 for x in [*maxima, final_score]):
            raise ValueError("Score metadata must be finite and nonnegative.")
        if not str(row["question"]).strip() or not str(row["desired_answer"]).strip():
            raise ValueError("Question and desired answer must be nonempty.")
        records.append({
            "row_index": index,
            **{key: str(row[key]) for key in ("question", "desired_answer", "student_answer")},
            "criteria": criteria, "criteria_score": scores,
            "criteria_max_score": maxima, "student_score": final_score,
        })
    except (ValueError, TypeError, SyntaxError) as error:
        errors.append(f"Row {index}: {error}")
if errors:
    raise ValueError(f"{len(errors)} invalid rows:\n" + "\n".join(errors[:20]))
print(f"All {len(records)} rows passed validation.")
score_disagreements = sum(
    not math.isclose(sum(s * m for s, m in zip(r["criteria_score"], r["criteria_max_score"])), r["student_score"])
    for r in records
)
print("Recorded final scores differing from criterion totals:", score_disagreements)

All 3778 rows passed validation.
Recorded final scores differing from criterion totals: 0


## Grading prompt

- Return only the ordered JSON array of binary decisions.
- Share one prompt builder between training and inference.

In [6]:
SYSTEM_PROMPT = (
    "You are a strict grading assistant.\n\n"
    "Your task is to evaluate whether a student's answer fulfills each grading criterion.\n\n"
    "For each criterion:\n"
    "- Output 1 if the criterion is fulfilled.\n"
    "- Output 0 if the criterion is not fulfilled.\n\n"
    "Evaluate each criterion independently.\n\n"
    "Return ONLY a valid JSON array containing 0 or 1.\n"
    "The output array must contain exactly one value for each criterion, "
    "in the same order as the criteria provided.\n\n"
    "Do not include explanations.\n"
    "Do not include markdown.\n"
    "Do not include any text before or after the JSON array."
)

print(SYSTEM_PROMPT)

You are a strict grading assistant.

Your task is to evaluate whether a student's answer fulfills each grading criterion.

For each criterion:
- Output 1 if the criterion is fulfilled.
- Output 0 if the criterion is not fulfilled.

Evaluate each criterion independently.

Return ONLY a valid JSON array containing 0 or 1.
The output array must contain exactly one value for each criterion, in the same order as the criteria provided.

Do not include explanations.
Do not include markdown.
Do not include any text before or after the JSON array.


## Build examples

- Retain structured fields for evaluation; do not parse fields back out of prompt text.
- Keep the assistant completion separate from the prompt.

In [7]:
def make_prompt(question, desired_answer, student_answer, criteria):
    criteria_text = "\n".join(f"{i}. {text}" for i, text in enumerate(criteria, 1))
    user_text = (
        f"Question:\n{question}\n\n"
        f"Desired answer:\n{desired_answer}\n\n"
        f"Student answer:\n{student_answer}\n\n"
        f"Criteria:\n{criteria_text}"
    )
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_text},
    ]


dataset = Dataset.from_list([
    {
        **row,
        "prompt": make_prompt(row["question"], row["desired_answer"], row["student_answer"], row["criteria"]),
        "completion": [{"role": "assistant", "content": json.dumps(row["criteria_score"])}],
    }
    for row in records
])
print(dataset[0]["prompt"])

[{'role': 'system', 'content': "You are a strict grading assistant.\n\nYour task is to evaluate whether a student's answer fulfills each grading criterion.\n\nFor each criterion:\n- Output 1 if the criterion is fulfilled.\n- Output 0 if the criterion is not fulfilled.\n\nEvaluate each criterion independently.\n\nReturn ONLY a valid JSON array containing 0 or 1.\nThe output array must contain exactly one value for each criterion, in the same order as the criteria provided.\n\nDo not include explanations.\nDo not include markdown.\nDo not include any text before or after the JSON array."}, {'role': 'user', 'content': 'Question:\nWhat is the role of a prototype program in problem solving?\n\nDesired answer:\nTo simulate the behaviour of portions of the desired software product.\n\nStudent answer:\nHigh risk problems are address in the prototype program to make sure that the program is feasible.  A prototype may also be used to show a company that the software can be possibly programmed.\n

## Split by question for each seed

- Shuffle unique stripped question strings with each seed in `TRAINING_SEEDS`, then repartition all three sets.
- Allocate approximately 80/10/10% of questions; row proportions can differ.
- Keep every row for a question together and require nonempty, disjoint train, validation, and test sets within each run.
- Save each run's question membership for reproducibility; selection rules and baseline share that run's held-out test set.


In [8]:
question_keys = [row["question"].strip() for row in records]
unique_questions = list(dict.fromkeys(question_keys))


def make_question_splits(split_seed):
    shuffled_questions = unique_questions.copy()
    random.Random(split_seed).shuffle(shuffled_questions)
    n_questions = len(shuffled_questions)
    train_end = int(n_questions * 0.80)
    validation_end = train_end + int(n_questions * 0.10)
    question_splits = {
        "train": shuffled_questions[:train_end],
        "validation": shuffled_questions[train_end:validation_end],
        "test": shuffled_questions[validation_end:],
    }
    if any(not questions for questions in question_splits.values()):
        raise ValueError("Too few unique questions for nonempty 80/10/10 splits.")
    assert sum(map(len, question_splits.values())) == len(set.union(
        *(set(questions) for questions in question_splits.values())
    )) == n_questions
    indices_by_question = defaultdict(list)
    for index, question in enumerate(question_keys):
        indices_by_question[question].append(index)
    # Preserve the seeded question order in training, validation, and test.
    splits = {
        name: dataset.select([
            index for question in questions for index in indices_by_question[question]
        ])
        for name, questions in question_splits.items()
    }
    assert sum(map(len, splits.values())) == len(dataset)
    return {"split_seed": split_seed, "question_splits": question_splits, "splits": splits}


seed_data = {seed: make_question_splits(seed) for seed in TRAINING_SEEDS}
for seed, data in seed_data.items():
    print(f"Seed {seed} question split:")
    for name, split in data["splits"].items():
        print(f"  {name}: {len(split)} rows / {len(data['question_splits'][name])} questions")


Seed 42 question split:
  train: 2980 rows / 76 questions
  validation: 409 rows / 9 questions
  test: 389 rows / 10 questions
Seed 123 question split:
  train: 2889 rows / 76 questions
  validation: 611 rows / 9 questions
  test: 278 rows / 10 questions
Seed 2026 question split:
  train: 3113 rows / 76 questions
  validation: 269 rows / 9 questions
  test: 396 rows / 10 questions


## Tokenizer

- Use the text tokenizer for both supervised examples and inference.
- Preserve the same chat template and disabled thinking mode throughout.

In [9]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
print("Tokenizer:", tokenizer.__class__.__name__)

Tokenizer: Qwen3_5Tokenizer


## Tokenize and verify completion masks

- Pre-tokenize each seed's training/validation examples; pass only token IDs and labels to TRL.
- Verify the generation prompt is an exact prefix of the full conversation.
- Mask prompt labels with `-100`; supervise the complete assistant turn, including its end token.
- Measure actual lengths before model loading; never silently truncate completions.

In [10]:
def tokenize_example(example):
    prompt_ids = tokenizer.apply_chat_template(
        example["prompt"], tokenize=True, return_dict=False,
        add_generation_prompt=True, enable_thinking=False,
    )
    input_ids = tokenizer.apply_chat_template(
        example["prompt"] + example["completion"], tokenize=True, return_dict=False,
        add_generation_prompt=False, enable_thinking=False,
    )
    boundary = len(prompt_ids)
    if input_ids[:boundary] != prompt_ids or boundary >= len(input_ids):
        raise ValueError(f"Invalid prompt/completion boundary at row {example['row_index']}.")
    return {"input_ids": input_ids, "labels": [-100] * boundary + input_ids[boundary:]}


for seed, data in seed_data.items():
    formatted_train, formatted_validation = (
        data["splits"][name].map(
            tokenize_example, remove_columns=data["splits"][name].column_names,
            desc=f"Seed {seed}: tokenize {name}",
        )
        for name in ("train", "validation")
    )
    lengths = [len(ids) for split in (formatted_train, formatted_validation) for ids in split["input_ids"]]
    measured_max_length = max(lengths)
    sequence_limit = MAX_SEQ_LENGTH if MAX_SEQ_LENGTH is not None else math.ceil(measured_max_length / 32) * 32
    if sequence_limit < measured_max_length:
        raise ValueError(
            f"Seed {seed}: {sum(length > sequence_limit for length in lengths)} examples exceed "
            f"{sequence_limit} tokens; longest is {measured_max_length}. Increase MAX_SEQ_LENGTH or use None."
        )
    data.update({
        "formatted_train": formatted_train, "formatted_validation": formatted_validation,
        "sequence_limit": sequence_limit,
    })
    print(f"Seed {seed} training/validation token lengths:", pd.Series(lengths).describe().round(1).to_dict())
    print("Sequence limit:", sequence_limit)
    print("Example supervised completion:", tokenizer.decode(
        [label for label in formatted_train[0]["labels"] if label != -100], skip_special_tokens=False,
    ))


Seed 42: tokenize train:   0%|          | 0/2980 [00:00<?, ? examples/s]

Seed 42: tokenize validation:   0%|          | 0/409 [00:00<?, ? examples/s]

Seed 42 training/validation token lengths: {'count': 3389.0, 'mean': 296.1, 'std': 108.5, 'min': 178.0, '25%': 221.0, '50%': 258.0, '75%': 331.0, 'max': 806.0}
Sequence limit: 832
Example supervised completion: [1, 1, 1]<|im_end|>



Seed 123: tokenize train:   0%|          | 0/2889 [00:00<?, ? examples/s]

Seed 123: tokenize validation:   0%|          | 0/611 [00:00<?, ? examples/s]

Seed 123 training/validation token lengths: {'count': 3500.0, 'mean': 295.5, 'std': 105.2, 'min': 178.0, '25%': 222.0, '50%': 266.0, '75%': 327.0, 'max': 806.0}
Sequence limit: 832
Example supervised completion: [1, 1, 1]<|im_end|>



Seed 2026: tokenize train:   0%|          | 0/3113 [00:00<?, ? examples/s]

Seed 2026: tokenize validation:   0%|          | 0/269 [00:00<?, ? examples/s]

Seed 2026 training/validation token lengths: {'count': 3382.0, 'mean': 296.6, 'std': 108.3, 'min': 179.0, '25%': 221.0, '50%': 259.0, '75%': 331.0, 'max': 806.0}
Sequence limit: 832
Example supervised completion: [1, 0]<|im_end|>



## Fresh model loading for each seed

- Define a loader here; the seed loop calls it once per run.
- NF4, double quantisation, BF16/FP16, SDPA, and one-GPU placement match the comparison notebooks.


In [11]:
def load_quantized_base_model():
    if not torch.cuda.is_available():
        raise RuntimeError("QLoRA training requires a CUDA GPU.")
    if not cc:
        raise RuntimeError("Install a C compiler with the dependency cell, then restart the kernel.")
    if int(os.environ.get("WORLD_SIZE", "1")) != 1:
        raise RuntimeError("This notebook's batch and validation schedule assume one training GPU.")
    compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=compute_dtype,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, quantization_config=bnb_config, device_map={"": 0},
        dtype=compute_dtype, attn_implementation="sdpa",
    )
    model.config.use_cache = False
    model.generation_config.pad_token_id = tokenizer.pad_token_id
    print("Loaded in 4-bit:", model.is_loaded_in_4bit, "| Compute dtype:", compute_dtype)
    return model


## Fresh LoRA adapters

- The seed is reset immediately before fresh model loading/adapter initialization.
- Use the same text attention/MLP projection targets as notebook 7, without changing rank or coverage.


In [12]:
def attach_fresh_lora(model):
    for param in model.parameters():
        param.requires_grad_(False)
    candidate_suffixes = {"q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"}
    target_modules = [
        name for name, module in model.named_modules()
        if name.startswith("model.layers.") and isinstance(module, torch.nn.Linear)
        and name.rsplit(".", 1)[-1] in candidate_suffixes
    ]
    if not target_modules:
        raise RuntimeError("No text projection targets found; inspect model.named_modules().")
    print("LoRA projection layers:", len(target_modules))
    model = get_peft_model(model, LoraConfig(
        r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
        bias="none", task_type="CAUSAL_LM", target_modules=target_modules,
    ))
    model.print_trainable_parameters()
    return model


## Shared grading helpers

- Validation, every saved adapter, and every baseline use the same prompt, parser, greedy decoding, and metric code.
- Pooled criterion metrics retain notebook 7's valid-output-only convention.
- Question-averaged exact accuracy and criterion accuracy include invalid arrays as failures.
- Question-averaged criterion F1 is the mean of each question's positive-class F1; invalid arrays count as criterion errors. Perfect all-negative questions score 1.
- Ground-truth labels are used only to calculate metrics; they are never supplied to generation.


In [13]:
def parse_prediction(raw_output, expected_count):
    recovered = False
    try:
        scores = json.loads(raw_output)
    except json.JSONDecodeError:
        candidates = re.findall(r"\[\s*[01](?:\s*,\s*[01])*\s*\]", raw_output)
        scores = next((
            parsed for candidate in reversed(candidates)
            if len(parsed := json.loads(candidate)) == expected_count
        ), None)
        recovered = scores is not None
    error = None
    if not isinstance(scores, list):
        error = "Output was not a JSON array."
    elif len(scores) != expected_count:
        error = f"Expected {expected_count} scores, received {len(scores)}."
    elif any(type(score) is not int or score not in (0, 1) for score in scores):
        error = "Scores must be integer 0 or 1."
    return {
        "valid": error is None, "raw_output": raw_output,
        "scores": scores if error is None else None,
        "error": error, "recovered": recovered,
    }


@torch.inference_mode()
def predict_criteria_scores(question, desired_answer, student_answer, criteria, prediction_model):
    inputs = tokenizer.apply_chat_template(
        make_prompt(question, desired_answer, student_answer, criteria),
        tokenize=True, return_dict=True, return_tensors="pt",
        add_generation_prompt=True, enable_thinking=False,
    ).to(prediction_model.get_input_embeddings().weight.device)
    generated = prediction_model.generate(
        **inputs, do_sample=False, use_cache=True,
        pad_token_id=tokenizer.pad_token_id,
        max_new_tokens=max(GENERATION_MIN_TOKENS, len(criteria) * GENERATION_TOKENS_PER_CRITERION),
    )
    answer_ids = generated[0, inputs["input_ids"].shape[-1]:]
    raw_output = tokenizer.decode(answer_ids, skip_special_tokens=True).strip()
    return parse_prediction(raw_output, len(criteria))


In [14]:
def summarize_question_metrics(rows):
    """Give each question equal weight; invalid arrays fail all their criteria."""
    grouped = defaultdict(list)
    for row in rows:
        grouped[row["question"].strip()].append(row)
    question_rows = []
    for question, answers in grouped.items():
        tp = fp = fn = criterion_correct = criterion_total = 0
        for answer in answers:
            truth = answer["ground_truth"]
            criterion_total += len(truth)
            if not answer["valid"]:
                # Count missing positive decisions as FN and missing negative decisions as FP.
                fn += sum(label == 1 for label in truth)
                fp += sum(label == 0 for label in truth)
                continue
            for actual, predicted in zip(truth, answer["prediction"]):
                criterion_correct += int(actual == predicted)
                tp += int(actual == predicted == 1)
                fp += int(actual == 0 and predicted == 1)
                fn += int(actual == 1 and predicted == 0)
        denominator = 2 * tp + fp + fn
        question_rows.append({
            "question": question,
            "examples": len(answers),
            "criteria": criterion_total,
            "invalid_output_rate": sum(not a["valid"] for a in answers) / len(answers),
            "exact_match_accuracy": sum(a["exact_match"] for a in answers) / len(answers),
            "per_criterion_accuracy": criterion_correct / criterion_total,
            # A perfectly graded all-negative question has F1=1, rather than undefined F1.
            "criterion_f1": 2 * tp / denominator if denominator else 1.0,
        })
    count = len(question_rows)
    summary = {"question_count": count}
    for name, column in (
        ("question_macro_exact_match_accuracy", "exact_match_accuracy"),
        ("question_macro_per_criterion_accuracy", "per_criterion_accuracy"),
        ("question_macro_criterion_f1", "criterion_f1"),
    ):
        summary[name] = sum(row[column] for row in question_rows) / count if count else 0.0
    return summary, question_rows


def evaluate_current_model(dataset, model_name, prediction_model, show_progress=False):
    """Generate predictions and calculate grading metrics for one model state."""
    criterion_true = []
    criterion_pred = []
    exact_correct = 0
    invalid_count = 0
    final_score_errors = []
    final_score_exact = 0
    final_score_within_one = 0
    valid_true_final_scores = []
    valid_predicted_final_scores = []
    rows = []

    for example in tqdm(dataset, desc=f"{model_name} grades", disable=not show_progress):
        true_scores = example["criteria_score"]
        max_scores = example["criteria_max_score"]
        true_final_score = example["student_score"]
        prediction = predict_criteria_scores(
            example["question"], example["desired_answer"],
            example["student_answer"], example["criteria"], prediction_model=prediction_model,
        )

        # Invalid arrays are failures for response-level exact matching.
        is_valid = prediction["valid"]
        pred_scores = prediction["scores"] if is_valid else None
        is_exact = bool(is_valid and pred_scores == true_scores)
        exact_correct += int(is_exact)
        invalid_count += int(not is_valid)

        # Criterion metrics require aligned arrays, so invalid arrays are omitted.
        if is_valid:
            criterion_true.extend(true_scores)
            criterion_pred.extend(pred_scores)

            # Convert binary criterion decisions into the final numeric score.
            predicted_final_score = sum(
                score * maximum for score, maximum in zip(pred_scores, max_scores)
            )
            score_error = abs(predicted_final_score - true_final_score)
            final_score_errors.append(score_error)
            final_score_exact += int(score_error == 0)
            final_score_within_one += int(score_error <= 1)
            valid_true_final_scores.append(true_final_score)
            valid_predicted_final_scores.append(predicted_final_score)
        else:
            predicted_final_score = None
            score_error = None

        rows.append({
            "model": model_name,
            "question": example["question"].strip(),
            "row_index": example["row_index"],
            "raw_output": prediction["raw_output"],
            "recovered": prediction["recovered"],
            "error": prediction["error"],
            "ground_truth": true_scores,
            "prediction": pred_scores if is_valid else prediction["raw_output"],
            "valid": is_valid,
            "exact_match": is_exact,
            "true_final_score": true_final_score,
            "predicted_final_score": predicted_final_score,
            "final_score_absolute_error": score_error,
        })

    total = len(dataset)
    precision, recall, f1, _ = (
        precision_recall_fscore_support(criterion_true, criterion_pred, average="binary", zero_division=0)
        if criterion_true else (float("nan"),) * 4
    )
    if len(valid_true_final_scores) > 1 and len(set(valid_true_final_scores)) > 1 and len(set(valid_predicted_final_scores)) > 1:
        final_score_pearson = float(np.corrcoef(valid_true_final_scores, valid_predicted_final_scores)[0, 1])
    else:
        final_score_pearson = float("nan")
    if valid_true_final_scores:
        final_score_qwk = float(cohen_kappa_score(
            valid_true_final_scores, valid_predicted_final_scores, weights="quadratic"
        ))
    else:
        final_score_qwk = float("nan")

    metrics = {
        "model": model_name,
        "examples": total,
        "valid_examples": total - invalid_count,
        "recovered_output_rate": sum(row["recovered"] for row in rows) / total if total else 0.0,
        "invalid_output_rate": invalid_count / total if total else 0.0,
        "exact_match_accuracy": exact_correct / total if total else 0.0,
        "per_criterion_accuracy": accuracy_score(criterion_true, criterion_pred) if criterion_true else float("nan"),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        # Invalid generations count as incorrect for the two score accuracies.
        "final_score_exact_accuracy": final_score_exact / total if total else 0.0,
        "final_score_within_one_accuracy": final_score_within_one / total if total else 0.0,
        # MAE, RMSE, Pearson, and QWK describe valid numeric predictions only.
        "final_score_mae": (
            sum(final_score_errors) / len(final_score_errors)
            if final_score_errors else float("nan")
        ),
        "final_score_rmse": (
            (sum(error ** 2 for error in final_score_errors) / len(final_score_errors)) ** 0.5
            if final_score_errors else float("nan")
        ),
        "final_score_pearson": final_score_pearson,
        "final_score_qwk": final_score_qwk,
    }
    question_summary, _ = summarize_question_metrics(rows)
    metrics.update(question_summary)
    return metrics, rows


## Shared loss-based training configuration

- Quarter-epoch evaluation/save schedule, optimizer, and effective batch size match notebook 7.
- LR reduction, early stopping, and Trainer's restored checkpoint all minimize `eval_loss`.
- Independent grading-based adapter snapshots do not alter those controls.


In [15]:
for seed, data in seed_data.items():
    microbatches_per_epoch = math.ceil(len(data["formatted_train"]) / TRAIN_BATCH_SIZE)
    optimizer_steps_per_epoch = math.ceil(microbatches_per_epoch / GRADIENT_ACCUMULATION_STEPS)
    data["optimizer_steps_per_epoch"] = optimizer_steps_per_epoch
    data["quarter_epoch_steps"] = max(1, round(optimizer_steps_per_epoch / VALIDATIONS_PER_EPOCH))
    print(f"Seed {seed}: {optimizer_steps_per_epoch} optimizer steps per epoch; "
          f"evaluate/save every {data['quarter_epoch_steps']} optimizer steps")


def make_training_args(run_dir, training_seed, data):
    quarter_epoch_steps = data["quarter_epoch_steps"]
    training_args = SFTConfig(
        output_dir=str(run_dir / "checkpoints"),
        seed=training_seed,
        data_seed=training_seed,
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        optim="paged_adamw_8bit",
        lr_scheduler_type="reduce_lr_on_plateau",
        lr_scheduler_kwargs={
            "mode": "min", "factor": LR_REDUCTION_FACTOR,
            "patience": LR_PATIENCE, "min_lr": MIN_LEARNING_RATE,
        },
        bf16=torch.cuda.is_bf16_supported(),
        fp16=not torch.cuda.is_bf16_supported(),
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        activation_offloading=False,
        loss_type="nll",
        completion_only_loss=True,
        max_length=data["sequence_limit"],
        packing=False,
        padding_free=False,
        logging_steps=max(1, quarter_epoch_steps // 2),
        save_strategy="steps",
        save_steps=quarter_epoch_steps,
        eval_strategy="steps",
        eval_steps=quarter_epoch_steps,
        prediction_loss_only=True,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        save_total_limit=3,
        report_to="none",
    )
    return training_args


Seed 42: 373 optimizer steps per epoch; evaluate/save every 93 optimizer steps
Seed 123: 362 optimizer steps per epoch; evaluate/save every 90 optimizer steps
Seed 2026: 390 optimizer steps per epoch; evaluate/save every 98 optimizer steps


## Independent checkpoint tracking

- Add generated validation metrics before Trainer logging and early-stopping callbacks.
- Save each rule's best adapter/tokenizer to a named folder outside rotating `checkpoints/checkpoint-*` directories.
- Strict improvements replace a rule's snapshot; ties keep the earlier checkpoint.
- Save selection metadata and validation history during training so completed evaluations remain inspectable if a later seed fails.
- Restore training mode and RNG state after generated grading validation.


In [16]:
class GradingMetricSFTTrainer(SFTTrainer):
    """Track several selection rules while Trainer continues to optimize eval_loss."""

    def __init__(self, *args, grading_validation_dataset, run_dir, training_seed, **kwargs):
        if not len(grading_validation_dataset):
            raise ValueError("Checkpoint selection requires a nonempty validation set.")
        self.grading_validation_dataset = grading_validation_dataset
        self.run_dir = Path(run_dir)
        self.training_seed = training_seed
        self.best_selections = {}
        self.validation_history = []
        super().__init__(*args, **kwargs)
        if self.args.world_size != 1:
            raise ValueError("This experiment supports one training GPU.")

    def preserve_best_adapters(self, metrics):
        for selection, (metric, maximize) in CHECKPOINT_SELECTIONS.items():
            value = float(metrics[metric])
            if not math.isfinite(value):
                raise ValueError(f"Nonfinite checkpoint selection metric: {metric}")
            previous = self.best_selections.get(selection)
            if previous is not None:
                improved = value > previous["value"] if maximize else value < previous["value"]
                if not improved:
                    continue
            adapter_dir = self.run_dir / selection
            self.model.save_pretrained(str(adapter_dir))
            tokenizer.save_pretrained(str(adapter_dir))
            record = {
                "selection": selection, "metric": metric, "maximize": maximize,
                "value": value, "step": self.state.global_step, "epoch": self.state.epoch,
                "training_seed": self.training_seed, "split_seed": self.training_seed,
                "adapter_dir": str(adapter_dir), "validation_metrics": dict(metrics),
            }
            self.best_selections[selection] = record
            (adapter_dir / "selection.json").write_text(json.dumps(record, indent=2) + "\n")
        (self.run_dir / "checkpoint_selections.json").write_text(
            json.dumps(self.best_selections, indent=2) + "\n"
        )

    def evaluation_loop(
        self, dataloader, description, prediction_loss_only=None,
        ignore_keys=None, metric_key_prefix="eval",
    ):
        was_training = self.model.training
        try:
            output = super().evaluation_loop(
                dataloader, description, prediction_loss_only=prediction_loss_only,
                ignore_keys=ignore_keys, metric_key_prefix=metric_key_prefix,
            )
            if metric_key_prefix != "eval":
                return output
            self.model.eval()
            # Isolate any RNG use during generation/saving from subsequent training.
            python_rng = random.getstate()
            numpy_rng = np.random.get_state()
            try:
                with torch.random.fork_rng(devices=[self.args.device.index or 0]):
                    with self.autocast_smart_context_manager():
                        grading_metrics, _ = evaluate_current_model(
                            self.grading_validation_dataset, "Validation",
                            prediction_model=self.model, show_progress=True,
                        )
                    output.metrics.update({
                        f"eval_grading_{key}": value
                        for key, value in grading_metrics.items() if key != "model"
                    })
                    self.preserve_best_adapters(output.metrics)
            finally:
                random.setstate(python_rng)
                np.random.set_state(numpy_rng)
            self.validation_history.append({
                "seed": self.training_seed, "step": self.state.global_step,
                "epoch": self.state.epoch,
                "learning_rate_before_plateau": self._get_learning_rate(),
                **output.metrics,
            })
            pd.DataFrame(self.validation_history).to_csv(
                self.run_dir / "validation_grading_metrics.csv", index=False,
            )
            return output
        finally:
            self.model.train(was_training)


## Check the collated loss mask

- Inspect a padded batch as seen by the trainer.
- Prompt and padding labels must be `-100`; completion tokens must remain supervised.

In [17]:
def verify_completion_masks(trainer):
    samples = [trainer.train_dataset[i] for i in range(min(2, len(trainer.train_dataset)))]
    batch = trainer.data_collator(samples)
    for index, sample in enumerate(samples):
        labels = batch["labels"][index]
        length = len(sample["labels"])
        assert labels[:length].tolist() == sample["labels"], "Collator changed the completion mask."
        assert (labels[length:] == -100).all(), "Padding contributes to loss."
        assert (labels[:length] == -100).any() and (labels[:length] != -100).any()
    print("Prompt/completion/padding masks verified.")


## Train and evaluate each seed

- Use each seed's own question split, tokenized examples, sequence limit, and quarter-epoch schedule.
- Fresh model/adapter/optimizer per seed; release GPU allocations before starting the next seed.
- Preserve all four adapters while training, before ordinary checkpoint rotation can remove their source steps.
- After training, evaluate each distinct selected step once, then evaluate the disabled-adapter baseline for this seed.
- Save metric CSVs, raw prediction JSONL, per-question CSVs, and per-seed comparison CSVs. Display the combined result tables after all configured seeds finish evaluation.
- Deltas are fine-tuned minus baseline: positive is better for accuracy/F1; negative is better for error metrics.

In [18]:
def release_training_storage(trainer):
    """Release optimizer/gradient storage after training or when cleaning up a failed run."""
    if trainer is None:
        return
    trainer.model.zero_grad(set_to_none=True)
    trainer.optimizer = None
    trainer.lr_scheduler = None
    trainer.accelerator.free_memory()


def clear_gpu_cache():
    gc.collect()
    torch.cuda.empty_cache()


def make_test_result(training_seed, selection, metrics, record=None):
    result = {
        "seed": training_seed, "selection": selection,
        **{f"test_{key}": value for key, value in metrics.items() if key != "model"},
    }
    if record is not None:
        result.update({
            "selected_step": record["step"], "selected_epoch": record["epoch"],
            "selection_metric": record["metric"], "selection_value": record["value"],
            "validation_loss": record["validation_metrics"]["eval_loss"],
            "validation_exact_accuracy": record["validation_metrics"]["eval_grading_exact_match_accuracy"],
        })
    return result


def save_evaluation_artifacts(run_dir, selection, metrics, rows):
    target = Path(run_dir) / "evaluation"
    target.mkdir(exist_ok=True)
    pd.DataFrame([metrics]).to_csv(target / f"{selection}_metrics.csv", index=False)
    pd.DataFrame(rows).to_json(target / f"{selection}_predictions.jsonl", orient="records", lines=True)
    _, question_rows = summarize_question_metrics(rows)
    pd.DataFrame(question_rows).to_csv(target / f"{selection}_per_question.csv", index=False)


def evaluate_seed_selections(model, run_dir, selections, training_seed, test_dataset):
    """Test saved selections only after training; baseline is evaluated for each seed."""
    model.gradient_checkpointing_disable()
    model.config.use_cache = True
    model.eval()
    results = []
    # Selection rules can pick the identical checkpoint; generate only once per step.
    evaluated_steps = {}
    for selection, record in selections.items():
        step = record["step"]
        if step not in evaluated_steps:
            adapter_name = "checkpoint_comparison"
            try:
                model.load_adapter(record["adapter_dir"], adapter_name=adapter_name, is_trainable=False)
                model.set_adapter(adapter_name)
                model.eval()
                metrics, rows = evaluate_current_model(
                    test_dataset, selection, prediction_model=model, show_progress=True,
                )
                evaluated_steps[step] = (metrics, rows)
            finally:
                model.set_adapter("default")
                if adapter_name in model.peft_config:
                    model.delete_adapter(adapter_name)
        cached_metrics, cached_rows = evaluated_steps[step]
        metrics = {**cached_metrics, "model": selection}
        rows = [{**row, "model": selection} for row in cached_rows]
        save_evaluation_artifacts(run_dir, selection, metrics, rows)
        results.append(make_test_result(training_seed, selection, metrics, record))
    model.eval()
    with model.disable_adapter():
        baseline_metrics, baseline_rows = evaluate_current_model(
            test_dataset, "baseline", prediction_model=model, show_progress=True,
        )
    save_evaluation_artifacts(run_dir, "baseline", baseline_metrics, baseline_rows)
    results.append(make_test_result(training_seed, "baseline", baseline_metrics))
    pd.DataFrame(results).to_csv(Path(run_dir) / "test_results.csv", index=False)
    return results


def train_and_evaluate_seed(training_seed, experiment_dir):
    run_dir = Path(experiment_dir) / f"seed_{training_seed}"
    run_dir.mkdir(exist_ok=False)
    data = seed_data[training_seed]
    model = trainer = None
    try:
        # Reset before model/adapter creation, not just before Trainer.train().
        set_seed(training_seed)
        model = load_quantized_base_model()
        model = attach_fresh_lora(model)
        trainer = GradingMetricSFTTrainer(
            model=model, args=make_training_args(run_dir, training_seed, data),
            train_dataset=data["formatted_train"], eval_dataset=data["formatted_validation"],
            grading_validation_dataset=data["splits"]["validation"].remove_columns(["completion"]),
            run_dir=run_dir, training_seed=training_seed,
            processing_class=tokenizer,
            callbacks=[EarlyStoppingCallback(
                early_stopping_patience=EARLY_STOPPING_PATIENCE,
                early_stopping_threshold=EARLY_STOPPING_THRESHOLD,
            )],
        )
        verify_completion_masks(trainer)
        train_result = trainer.train()
        (run_dir / "training_metrics.json").write_text(json.dumps(train_result.metrics, indent=2) + "\n")
        selections = dict(trainer.best_selections)
        if set(selections) != set(CHECKPOINT_SELECTIONS):
            raise RuntimeError("Training did not produce every expected checkpoint selection.")
        selection_table = pd.DataFrame([
            {key: record[key] for key in ("selection", "metric", "value", "step", "epoch")}
            for record in selections.values()
        ])
        print(f"Seed {training_seed}: preserved validation-selected adapters")
        display(selection_table)
        model = trainer.model
        release_training_storage(trainer)
        trainer = None
        clear_gpu_cache()
        return evaluate_seed_selections(
            model, run_dir, selections, training_seed, data["splits"]["test"],
        ) if RUN_TEST_EVALUATION else []
    finally:
        release_training_storage(trainer)
        trainer = model = None
        clear_gpu_cache()


def validate_seed_results(results_df, expected_seeds):
    """Require exactly one baseline and each selection for every expected seed."""
    expected = {
        (seed, selection)
        for seed in expected_seeds for selection in ("baseline", *CHECKPOINT_SELECTIONS)
    }
    if results_df.empty or not {"seed", "selection"}.issubset(results_df.columns):
        raise ValueError("Seed evaluation results are empty or missing identifying columns.")
    if results_df.duplicated(["seed", "selection"]).any():
        raise ValueError("Duplicate evaluation results for a seed/selection pair.")
    actual = set(results_df[["seed", "selection"]].itertuples(index=False, name=None))
    if actual != expected:
        raise ValueError(f"Incomplete seed evaluation results. Missing: {sorted(expected - actual)}; unexpected: {sorted(actual - expected)}")
    if not any(column.startswith("test_") for column in results_df.columns):
        raise ValueError("No test evaluation metrics found.")


def format_metric_comparison(metrics_by_selection, delta_label=None):
    """Format metric rows for per-seed, mean, and standard-deviation CSV exports."""
    table = metrics_by_selection.T.copy()
    table.index = table.index.str.removeprefix("test_")
    table = table.reindex(columns=["baseline", *CHECKPOINT_SELECTIONS])
    if delta_label is not None:
        for selection in CHECKPOINT_SELECTIONS:
            table[f"{selection} - {delta_label}"] = table[selection] - table["baseline"]
    return table


def build_seed_comparison(results_df, training_seed):
    subset = results_df.loc[results_df["seed"] == training_seed]
    validate_seed_results(subset, [training_seed])
    columns = [column for column in subset.columns if column.startswith("test_")]
    return format_metric_comparison(subset.set_index("selection")[columns], "baseline")


def build_average_comparison(results_df):
    validate_seed_results(results_df, TRAINING_SEEDS)
    columns = [column for column in results_df.columns if column.startswith("test_")]
    grouped = results_df.groupby("selection")[columns]
    means = format_metric_comparison(grouped.mean(), "average baseline")
    stds = format_metric_comparison(grouped.std(ddof=1))
    return means, stds


# Keep the complete metric schema and column order of the 7-series notebooks.
SERIES7_TEST_METRICS = [
    "test_examples",
    "test_valid_examples",
    "test_recovered_output_rate",
    "test_invalid_output_rate",
    "test_exact_match_accuracy",
    "test_per_criterion_accuracy",
    "test_precision",
    "test_recall",
    "test_f1",
    "test_final_score_exact_accuracy",
    "test_final_score_within_one_accuracy",
    "test_final_score_mae",
    "test_final_score_rmse",
    "test_final_score_pearson",
    "test_final_score_qwk",
]


def build_evaluation_tables(all_seed_results_df):
    """Show baseline/QLoRA pairs by seed and each model's mean across seeds."""
    validate_seed_results(all_seed_results_df, TRAINING_SEEDS)
    missing_metrics = sorted(set(SERIES7_TEST_METRICS) - set(all_seed_results_df.columns))
    if missing_metrics:
        raise ValueError(f"Missing 7-series test metrics: {missing_metrics}")
    # Include all 7-series metrics first, followed by the additional question-level metrics.
    metric_columns = [
        *SERIES7_TEST_METRICS,
        *(column for column in all_seed_results_df.columns
          if column.startswith("test_") and column not in SERIES7_TEST_METRICS),
    ]
    # Exactly two rows per seed: baseline, then the lowest-validation-loss QLoRA adapter.
    row_order = pd.MultiIndex.from_product(
        [TRAINING_SEEDS, ["baseline", "best_loss"]], names=["seed", "selection"],
    )
    ordered_results = (
        all_seed_results_df[["seed", "selection", *metric_columns]]
        .set_index(["seed", "selection"]).reindex(row_order).reset_index()
    )
    metric_names = {column: column.removeprefix("test_") for column in metric_columns}
    results_df = ordered_results[metric_columns].rename(columns=metric_names)
    results_df.insert(0, "Model", [
        f"{'Baseline' if selection == 'baseline' else 'QLoRA'} seed {seed}"
        for seed, selection in row_order
    ])
    # Each seed contributes equally; average only test metrics at their original precision.
    # Pandas skips undefined (NaN) values, as in the existing CSV means.
    average_results_df = (
        ordered_results.groupby("selection", sort=False)[metric_columns].mean()
        .reindex(["baseline", "best_loss"])
        .rename(columns=metric_names).reset_index(drop=True)
    )
    average_results_df.insert(0, "Model", ["Baseline average", "QLoRA average"])
    return results_df, average_results_df


def display_metric_table(table, title):
    print(title)
    with pd.option_context(
        "display.max_rows", None, "display.max_columns", None,
        "display.float_format", "{:.4f}".format,
    ):
        display(table)


In [19]:
all_seed_results = []
all_seed_results_df = pd.DataFrame()
experiment_dir = None
evaluation_completed = False
if RUN_TRAINING:
    experiment_dir = Path(OUTPUT_DIR) / datetime.now(timezone.utc).strftime("run-%Y%m%dT%H%M%S-%fZ")
    experiment_dir.mkdir(parents=True, exist_ok=False)
    experiment_manifest = {
        "model": MODEL_ID, "training_seeds": TRAINING_SEEDS,
        "split_seed_policy": "same as training seed",
        "question_split_fractions": {"train": 0.8, "validation": 0.1, "test": 0.1},
        "dataset_sha256": hashlib.sha256(Path(DATASET_PATH).read_bytes()).hexdigest(),
        "lora_rank": LORA_R, "lora_alpha": LORA_ALPHA, "learning_rate": LEARNING_RATE,
        "lora_dropout": LORA_DROPOUT, "weight_decay": WEIGHT_DECAY, "max_epochs": NUM_EPOCHS,
        "train_batch_size": TRAIN_BATCH_SIZE, "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
        "training_control_metric": "eval_loss", "checkpoint_selections": CHECKPOINT_SELECTIONS,
        "package_versions": {name: version(name) for name in (
            "torch", "transformers", "trl", "peft", "accelerate", "bitsandbytes", "datasets",
        )},
        "seed_splits": {
            str(seed): {
                "split_seed": data["split_seed"],
                "question_splits": data["question_splits"],
                "row_counts": {name: len(split) for name, split in data["splits"].items()},
                "sequence_limit": data["sequence_limit"],
                "optimizer_steps_per_epoch": data["optimizer_steps_per_epoch"],
                "quarter_epoch_steps": data["quarter_epoch_steps"],
            }
            for seed, data in seed_data.items()
        },
    }
    (experiment_dir / "experiment.json").write_text(json.dumps(experiment_manifest, indent=2) + "\n")
    print("Experiment directory:", experiment_dir)
    for run_number, training_seed in enumerate(TRAINING_SEEDS, start=1):
        print(f"Starting seed {training_seed} ({run_number}/{len(TRAINING_SEEDS)})")
        seed_results = train_and_evaluate_seed(training_seed, experiment_dir)
        all_seed_results.extend(seed_results)
        if seed_results:
            all_seed_results_df = pd.DataFrame(all_seed_results)
            all_seed_results_df.to_csv(experiment_dir / "all_seed_test_results.csv", index=False)
            comparison = build_seed_comparison(all_seed_results_df, training_seed)
            comparison.to_csv(experiment_dir / f"seed_{training_seed}_comparison_to_baseline.csv")
    if RUN_TEST_EVALUATION:
        validate_seed_results(all_seed_results_df, TRAINING_SEEDS)
        evaluation_completed = True
else:
    print("Data checks complete. Set RUN_TRAINING=True to run the sequential seed experiment.")


Experiment directory: qwen35-9b-grading-checkpoint-comparison/run-20261007T001544-924291Z
Starting seed 42 (1/3)


Loading weights:   0%|          | 0/427 [00:00<?, ?it/s]

Loaded in 4-bit: True | Compute dtype: torch.bfloat16
LoRA projection layers: 128


trainable params: 58,195,968 || all params: 9,011,999,232 || trainable%: 0.6458


Truncating train dataset:   0%|          | 0/2980 [00:00<?, ? examples/s]

...

Seed 2026: preserved validation-selected adapters


,selection,metric,value,step,epoch
0,best_loss,eval_loss,0.081187,784,2.010279
1,best_exact_accuracy,eval_grading_exact_match_accuracy,0.736059,490,1.256987
2,best_question_exact_accuracy,eval_grading_question_macro_exact_match_accuracy,0.719460,490,1.256987
3,best_question_f1,eval_grading_question_macro_criterion_f1,0.852931,784,2.010279


best_loss grades:   0%|          | 0/396 [00:00<?, ?it/s]

best_exact_accuracy grades:   0%|          | 0/396 [00:00<?, ?it/s]

baseline grades:   0%|          | 0/396 [00:00<?, ?it/s]

## Final evaluation results across testing seeds

- **Table 1 — Performance Across Testing Seeds:** Baseline seed 42, QLoRA seed 42, Baseline seed 123, QLoRA seed 123, Baseline seed 2026, and QLoRA seed 2026, in that order by default.
- **Table 2 — Average Performance Across Testing Seeds:** Baseline average and QLoRA average, calculated separately with equal weight for each seed. Undefined (`NaN`) values are skipped; a metric undefined for every seed remains `NaN`.
- The QLoRA rows display the lowest-validation-loss checkpoint (`best_loss`), matching the 7-series notebooks.
- Each row includes all 15 metrics from the 7-series notebooks in the same order: examples, valid_examples, recovered_output_rate, invalid_output_rate, exact_match_accuracy, per_criterion_accuracy, precision, recall, f1, final_score_exact_accuracy, final_score_within_one_accuracy, final_score_mae, final_score_rmse, final_score_pearson, and final_score_qwk. Append the four question-level metrics from this notebook. Require every original metric to be present in both tables; exclude seeds and checkpoint/validation metadata from averages. Display floating-point values to four decimal places without rounding stored results.
- Verify that every configured seed has exactly one baseline and one result for each selection before displaying the tables.
- Held-out questions change across seeds. Existing CSV exports retain all checkpoint selections, differences from baseline, and sample standard deviations (`ddof=1`), which reflect both training randomness and question-split variation.
- Outputs remain `all_seed_test_results.csv`, `seed_*_comparison_to_baseline.csv`, `average_comparison_to_baseline.csv`, and `across_seed_standard_deviations.csv`. Validation history and checkpoint metadata remain in each seed's run directory.



In [20]:
results_df = pd.DataFrame()
average_results_df = pd.DataFrame()
if evaluation_completed:
    # Preserve the existing comparison/statistics exports at their original precision.
    average_comparison, standard_deviations = build_average_comparison(all_seed_results_df)
    average_comparison.to_csv(experiment_dir / "average_comparison_to_baseline.csv")
    standard_deviations.to_csv(experiment_dir / "across_seed_standard_deviations.csv")
    results_df, average_results_df = build_evaluation_tables(all_seed_results_df)
    display_metric_table(results_df, "Table 1: Performance Across Testing Seeds")
    display_metric_table(average_results_df, "Table 2: Average Performance Across Testing Seeds")
else:
    print("Result tables require completed test evaluation for every configured seed.")



Table 1: Performance Across Testing Seeds


,Model,examples,valid_examples,recovered_output_rate,invalid_output_rate,exact_match_accuracy,per_criterion_accuracy,precision,recall,f1,final_score_exact_accuracy,final_score_within_one_accuracy,final_score_mae,final_score_rmse,final_score_pearson,final_score_qwk,question_count,question_macro_exact_match_accuracy,question_macro_per_criterion_accuracy,question_macro_criterion_f1
0,Baseline seed 42,389,389,0.0000,0.0000,0.7404,0.8686,0.8775,0.8845,0.8810,0.7404,0.9666,0.3085,0.6840,0.8083,0.8013,10,0.7593,0.8706,0.8651
1,QLoRA seed 42,389,389,0.0000,0.0000,0.7815,0.8938,0.8649,0.9562,0.9082,0.7943,0.9897,0.2237,0.5390,0.8772,0.8681,10,0.8065,0.8967,0.8988
2,Baseline seed 123,278,278,0.0000,0.0000,0.6547,0.7892,0.8166,0.7973,0.8068,0.6619,0.9676,0.3921,0.7467,0.7189,0.7180,10,0.6529,0.7792,0.7563
3,QLoRA seed 123,278,278,0.0000,0.0000,0.7194,0.8340,0.8127,0.9088,0.8581,0.7338,0.9820,0.2914,0.5968,0.8318,0.8239,10,0.7203,0.8240,0.8150
4,Baseline seed 2026,396,396,0.0000,0.0000,0.6970,0.8425,0.8540,0.8313,0.8425,0.6995,0.9798,0.3207,0.6009,0.7706,0.7694,10,0.6614,0.7843,0.7863
5,QLoRA seed 2026,396,396,0.0000,0.0000,0.7753,0.8864,0.8889,0.8867,0.8878,0.7803,0.9949,0.2247,0.4846,0.8494,0.8488,10,0.7406,0.8280,0.8353


Table 2: Average Performance Across Testing Seeds


,Model,examples,valid_examples,recovered_output_rate,invalid_output_rate,exact_match_accuracy,per_criterion_accuracy,precision,recall,f1,final_score_exact_accuracy,final_score_within_one_accuracy,final_score_mae,final_score_rmse,final_score_pearson,final_score_qwk,question_count,question_macro_exact_match_accuracy,question_macro_per_criterion_accuracy,question_macro_criterion_f1
0,Baseline average,354.3333,354.3333,0.0000,0.0000,0.6973,0.8334,0.8493,0.8377,0.8434,0.7006,0.9713,0.3404,0.6772,0.7659,0.7629,10.0000,0.6912,0.8114,0.8026
1,QLoRA average,354.3333,354.3333,0.0000,0.0000,0.7587,0.8714,0.8555,0.9172,0.8847,0.7695,0.9889,0.2466,0.5401,0.8528,0.8469,10.0000,0.7558,0.8496,0.8497


## Inspect persisted predictions

- Every seed and selection has raw test predictions and per-question results under its `evaluation` folder.
- Preview up to 20 predictions from the first configured seed's `best_loss` selection, reading only the first JSONL chunk.

In [21]:
if evaluation_completed:
    # All predictions remain on disk; show a small sample without retaining GPU models.
    inspection_path = experiment_dir / f"seed_{TRAINING_SEEDS[0]}" / "evaluation" / "best_loss_predictions.jsonl"
    with pd.read_json(inspection_path, lines=True, chunksize=20) as prediction_chunks:
        display(next(prediction_chunks, pd.DataFrame()))

,model,question,row_index,raw_output,recovered,error,ground_truth,prediction,valid,exact_match,true_final_score,predicted_final_score,final_score_absolute_error
0,best_loss,What is the main difference between a string o...,1834,[1],False,NaN,[1],[1],True,True,1,1,0
1,best_loss,What is the main difference between a string o...,1835,[0],False,NaN,[0],[0],True,True,0,0,0
2,best_loss,What is the main difference between a string o...,1836,[1],False,NaN,[1],[1],True,True,1,1,0
3,best_loss,What is the main difference between a string o...,1837,[1],False,NaN,[0],[1],True,False,0,1,1
4,best_loss,What is the main difference between a string o...,1838,[0],False,NaN,[0],[0],True,True,0,0,0
5,best_loss,What is the main difference between a string o...,1839,[0],False,NaN,[0],[0],True,True,0,0,0
6,best_loss,What is the main difference between a string o...,1840,[1],False,NaN,[1],[1],True,True,1,1,0
7,best_loss,What is the main difference between a string o...,1841,[1],False,NaN,[1],[1],True,True,1,1,0
8,best_loss,What is the main difference between a string o...,1842,[0],False,NaN,[0],[0],True,True,0,0,0
9,best_loss,What is the main difference between a string o...,1843,[0],False,NaN,[0],[0],True,True,0,0,0


## Optional upload of one evaluated adapter

- Upload is disabled by default; choose `UPLOAD_SEED` and `UPLOAD_SELECTION` explicitly.
- Upload only that adapter's folder, not the full experiment or every seed.
- Test results do not automatically choose which adapter is published.

In [22]:
if RUN_HF_UPLOAD:
    if not evaluation_completed or experiment_dir is None:
        raise RuntimeError("Complete all training and held-out evaluations before uploading.")
    if UPLOAD_SEED not in TRAINING_SEEDS or UPLOAD_SELECTION not in CHECKPOINT_SELECTIONS:
        raise ValueError("Choose an explicitly configured upload seed and selection.")
    upload_dir = experiment_dir / f"seed_{UPLOAD_SEED}" / UPLOAD_SELECTION
    if not (upload_dir / "adapter_config.json").is_file():
        raise RuntimeError(f"Selected adapter is missing: {upload_dir}")
    login(token=hf_token)
    upload_result = upload_folder(
        folder_path=str(upload_dir), repo_id=HF_REPO_ID, repo_type="model",
        commit_message=f"Upload evaluated Qwen3.5-9B adapter: seed {UPLOAD_SEED}, {UPLOAD_SELECTION}",
    )
    print("Uploaded selected adapter to:", upload_result)
else:
    print("Upload skipped. Configure UPLOAD_SEED/UPLOAD_SELECTION and RUN_HF_UPLOAD to publish one adapter.")

Upload skipped. Configure UPLOAD_SEED/UPLOAD_SELECTION and RUN_HF_UPLOAD to publish one adapter.
